# 第 3.5 课：Agent 第一性原理与综合系统实验

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TardisBooo/agenttect-notebooks/blob/main/notebooks/student/lessons/03_5_agent_rag_research_synthesis.ipynb)

> **课程核心目标**：梳理完整 Agent 系统的“循环（Loop）、上下文（Context）与 Harness（评测装备）”。  
> 总结 Agent、RAG 与 Research 前三课的核心职责，从零实现一个极简但工业标准的 Coding Agent 运行时，并完成确定性防御检查。

## 1. 实验准备与环境初始化
在 Colab 沙箱中建立纯净的本地工作空间（Workspace），确保所有工具操作严格受限于当前沙箱目录。

In [ ]:
import os
import sys
import json
import hashlib
from pathlib import Path
from typing import Any, Dict, List, Optional

# 创建独立工作空间
WORKSPACE = Path("./sandbox_workspace").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)

# 写入初始化测试文件
(WORKSPACE / "A.txt").write_text("Hello Agent World! Version 1.0", encoding="utf-8")
(WORKSPACE / "config.json").write_text(json.dumps({"env": "sandbox", "max_tokens": 4096}), encoding="utf-8")

print(f"Workspace initialized at: {WORKSPACE}")
print(f"Files: {[p.name for p in WORKSPACE.iterdir()]}")

## 2. 核心工具集：沙箱安全与文件操作
实现三大基础工具：
1. `list_files_tool`: 枚举工作区文件；
2. `read_file_tool`: 安全读取指定文件；
3. `edit_file_tool`: 精确文本替换与新文件写入。

In [ ]:
def resolve_safe_path(rel_path: str) -> Path:
    """防御路径遍历攻击（Path Traversal），确保只能访问工作区内部文件"""
    target = (WORKSPACE / Path(rel_path).expanduser()).resolve()
    if not target.is_relative_to(WORKSPACE):
        raise PermissionError(f"Path '{rel_path}' escapes sandbox workspace!")
    return target

def list_files_tool(path: str = ".") -> Dict[str, Any]:
    """列出工作区中的所有文件"""
    target = resolve_safe_path(path)
    files = [
        {"name": item.name, "is_dir": item.is_dir(), "size": item.stat().st_size}
        for item in sorted(target.iterdir())
    ]
    return {"status": "ok", "count": len(files), "files": files}

def read_file_tool(filename: str) -> Dict[str, Any]:
    """安全读取文件全部内容"""
    target = resolve_safe_path(filename)
    if not target.exists() or not target.is_file():
        return {"status": "error", "message": f"File '{filename}' not found"}
    return {"status": "ok", "content": target.read_text(encoding="utf-8")}

def edit_file_tool(filename: str, old_str: str, new_str: str) -> Dict[str, Any]:
    """精准字符串替换，若 old_str 为空则全量新建或覆盖"""
    target = resolve_safe_path(filename)
    if old_str == "":
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(new_str, encoding="utf-8")
        return {"status": "ok", "action": "created_or_overwritten"}
    if not target.exists():
        return {"status": "error", "message": f"File '{filename}' not found"}
    content = target.read_text(encoding="utf-8")
    if old_str not in content:
        return {"status": "error", "message": "Target string not found for replacement"}
    updated = content.replace(old_str, new_str, 1)
    target.write_text(updated, encoding="utf-8")
    return {"status": "ok", "action": "replaced_text"}

# 验证工具
assert list_files_tool()["count"] == 2
assert "Hello Agent" in read_file_tool("A.txt")["content"]
edit_res = edit_file_tool("A.txt", "Version 1.0", "Version 2.0")
assert edit_res["status"] == "ok"
assert "Version 2.0" in read_file_tool("A.txt")["content"]
print("Safety file tools verified successfully!")

## 3. Harness 防线：无进展哈希状态检测器 (No-Progress Stagnation Detector)
大模型在面对报错时常常陷入自回归偏执死循环。Harness 必须使用**物理指纹哈希**掐断无效空转！

In [ ]:
class StagnationDetector:
    """无进展状态哈希检测器"""
    def __init__(self, patience: int = 3):
        self.patience = patience
        self.history_hashes: List[str] = []

    def compute_state_hash(self, workspace_path: Path) -> str:
        """提取工作区所有文件的名称和 MD5 摘要并计算整体系统指纹"""
        items = []
        for p in sorted(workspace_path.rglob("*")):
            if p.is_file():
                digest = hashlib.md5(p.read_bytes()).hexdigest()
                items.append((str(p.relative_to(workspace_path)), digest))
        serialized = json.dumps(items, sort_keys=True)
        return hashlib.sha256(serialized.encode("utf-8")).hexdigest()

    def step(self, workspace_path: Path) -> bool:
        """
        记录当前状态哈希，如果连续 patience 轮哈希未发生实质变化，
        返回 True（触发熔断拦截），否则返回 False（允许继续执行）。
        """
        h = self.compute_state_hash(workspace_path)
        self.history_hashes.append(h)
        if len(self.history_hashes) >= self.patience:
            recent = self.history_hashes[-self.patience:]
            if len(set(recent)) == 1:
                return True # Stagnation detected! 停滞熔断
        return False

# 验证检测器
detector = StagnationDetector(patience=3)
assert detector.step(WORKSPACE) is False
assert detector.step(WORKSPACE) is False
assert detector.step(WORKSPACE) is True # 第三轮无进展，熔断！
print("StagnationDetector verified: Successfully trapped infinite loops!")

## 4. 学生实战任务：构建最小自主代理循环 (Autonomous Agent Loop)
请补全下方 `run_agent_loop` 函数中带有 `# TODO` 的部分，实现模型意图解析、工具分发和最大步数防御保护。

In [ ]:
def run_agent_loop(
    task_instructions: str,
    simulated_steps: List[Dict[str, Any]],
    max_steps: int = 5
) -> Dict[str, Any]:
    """
    执行最小 Agent 循环：
    - 接收步骤指令
    - 分发工具操作 (read_file / edit_file / list_files)
    - 利用 StagnationDetector 阻止停滞空转
    - 返回执行总结报告
    """
    detector = StagnationDetector(patience=3)
    executed_actions = []
    is_stagnated = False
    
    for step_idx, step in enumerate(simulated_steps):
        if step_idx >= max_steps:
            break
            
        tool_name = step.get("tool")
        params = step.get("params", {})
        
        # TODO: 根据 tool_name 分发调用对应的工具
        # 1. 'read_file' -> read_file_tool(params['filename'])
        # 2. 'edit_file' -> edit_file_tool(params['filename'], params['old_str'], params['new_str'])
        # 3. 'list_files' -> list_files_tool(params.get('path', '.'))
        result = {}
        if tool_name == "read_file":
            result = read_file_tool(params.get("filename", ""))
        elif tool_name == "edit_file":
            result = edit_file_tool(params.get("filename", ""), params.get("old_str", ""), params.get("new_str", ""))
        elif tool_name == "list_files":
            result = list_files_tool(params.get("path", "."))
            
        executed_actions.append({"step": step_idx, "tool": tool_name, "result": result})
        
        # 检查是否发生状态停滞
        if detector.step(WORKSPACE):
            is_stagnated = True
            break
            
    return {
        "total_steps": len(executed_actions),
        "is_stagnated": is_stagnated,
        "executed_actions": executed_actions
    }

print("run_agent_loop skeleton ready.")

## 5. 100 分自动评分与回归验收
运行下方测试用例，检验安全沙箱、工具分发和死循环熔断三大能力。

In [ ]:
def test_assignment_100_points():
    score = 0
    
    # Case 1: 沙箱越权路径拦截 (25分)
    try:
        resolve_safe_path("../../etc/passwd")
        print("❌ Case 1 失败：未能拦截越权路径")
    except PermissionError:
        score += 25
        print("✅ Case 1 通过：安全拦截路径穿越 (25/25)")
        
    # Case 2: 正常文件读写与精准替换 (25分)
    edit_file_tool("test_demo.txt", "", "Initial content")
    r1 = read_file_tool("test_demo.txt")
    edit_file_tool("test_demo.txt", "Initial", "Refactored")
    r2 = read_file_tool("test_demo.txt")
    if r1["content"] == "Initial content" and r2["content"] == "Refactored content":
        score += 25
        print("✅ Case 2 通过：精准文件读写与替换 (25/25)")
    else:
        print("❌ Case 2 失败：文件内容替换不符合预期")
        
    # Case 3: 死循环状态停滞熔断 (25分)
    dummy_steps = [
        {"tool": "read_file", "params": {"filename": "A.txt"}},
        {"tool": "read_file", "params": {"filename": "A.txt"}},
        {"tool": "read_file", "params": {"filename": "A.txt"}},
        {"tool": "read_file", "params": {"filename": "A.txt"}},
    ]
    res = run_agent_loop("Keep reading", dummy_steps, max_steps=10)
    if res["is_stagnated"] and res["total_steps"] == 3:
        score += 25
        print("✅ Case 3 通过：连续 3 次无进展成功熔断 (25/25)")
    else:
        print(f"❌ Case 3 失败：未能在第 3 步熔断，实际步数 {res['total_steps']}")
        
    # Case 4: 综合推进任务正常完成 (25分)
    progression_steps = [
        {"tool": "edit_file", "params": {"filename": "step1.txt", "old_str": "", "new_str": "ok1"}},
        {"tool": "edit_file", "params": {"filename": "step2.txt", "old_str": "", "new_str": "ok2"}},
    ]
    res2 = run_agent_loop("Progressive tasks", progression_steps, max_steps=10)
    if not res2["is_stagnated"] and res2["total_steps"] == 2:
        score += 25
        print("✅ Case 4 通过：推进式任务无误报中断 (25/25)")
    else:
        print("❌ Case 4 失败：有效任务被意外熔断")
        
    print("-" * 40)
    print(f"🎉 最终总得分：{score} / 100")
    return score

test_assignment_100_points()